# HR Employees — Exploratory Data Analysis (EDA)

**Project:** HR Employee Analysis  
**Dataset:** `HR_Employee_Analysis_Dataset.xlsx`

### Objectives
- Understand the HR workforce
- Clean and validate the data
- Calculate important HR KPIs
- Analyze salary, departments, roles and demographics
- Study performance, satisfaction, overtime and absenteeism
- Analyze employee attrition
- Identify hiring trends
- Produce business insights for a Power BI dashboard


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully")


In [ ]:
# 2. Load the HR Excel dataset
# Put HR_Employee_Analysis_Dataset.xlsx in the same folder as this notebook.

FILE_PATH = "HR_Employee_Analysis_Dataset.xlsx"

df = pd.read_excel(FILE_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 3. Dataset information
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)


In [ ]:
# 4. Data quality checks
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Employee IDs:", df["Employee_ID"].duplicated().sum())


In [ ]:
# 5. Clean data types and text
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"], errors="coerce")

text_columns = [
    "Gender", "Department", "Job_Role",
    "Education", "Overtime", "Attrition", "Location"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Cleaning completed.")
display(df.head())


## 6. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)


## 7. HR KPI Dashboard Summary

In [ ]:
kpis = {
    "Total Employees": len(df),
    "Average Age": df["Age"].mean(),
    "Average Salary": df["Salary"].mean(),
    "Average Experience": df["Experience_Years"].mean(),
    "Average Performance": df["Performance_Rating"].mean(),
    "Average Satisfaction": df["Job_Satisfaction"].mean(),
    "Average Work-Life Balance": df["Work_Life_Balance"].mean(),
    "Average Training Hours": df["Training_Hours"].mean(),
    "Average Absenteeism Days": df["Absenteeism_Days"].mean(),
    "Attrition Rate %": df["Attrition"].eq("Yes").mean() * 100,
    "Overtime Rate %": df["Overtime"].eq("Yes").mean() * 100
}

kpi_df = pd.DataFrame({
    "KPI": list(kpis.keys()),
    "Value": list(kpis.values())
})

display(kpi_df)


## 8. Workforce Demographics

In [ ]:
# Gender distribution
gender_summary = df["Gender"].value_counts().reset_index()
gender_summary.columns = ["Gender", "Employees"]
display(gender_summary)

plt.figure(figsize=(7,4))
sns.countplot(data=df, x="Gender")
plt.title("Employees by Gender")
plt.xlabel("Gender")
plt.ylabel("Employees")
plt.tight_layout()
plt.show()


In [ ]:
# Age distribution
plt.figure(figsize=(9,5))
sns.histplot(df["Age"], bins=15, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Employees")
plt.tight_layout()
plt.show()


In [ ]:
# Age groups
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[20, 24, 29, 34, 39, 49, 100],
    labels=["21-24", "25-29", "30-34", "35-39", "40-49", "50+"]
)

age_summary = df.groupby("Age_Group", observed=False).agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Performance=("Performance_Rating", "mean"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).reset_index()

display(age_summary.round(2))


## 9. Department Analysis

In [ ]:
department_summary = df.groupby("Department").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Experience=("Experience_Years", "mean"),
    Avg_Performance=("Performance_Rating", "mean"),
    Avg_Satisfaction=("Job_Satisfaction", "mean"),
    Avg_Work_Life_Balance=("Work_Life_Balance", "mean"),
    Avg_Training_Hours=("Training_Hours", "mean"),
    Avg_Absenteeism=("Absenteeism_Days", "mean"),
    Attrition_Count=("Attrition", lambda x: x.eq("Yes").sum()),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).sort_values("Employees", ascending=False)

display(department_summary.round(2))


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=df, y="Department", order=df["Department"].value_counts().index)
plt.title("Employees by Department")
plt.xlabel("Employees")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


In [ ]:
dept_salary = df.groupby("Department")["Salary"].mean().sort_values(ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(x=dept_salary.values, y=dept_salary.index)
plt.title("Average Salary by Department")
plt.xlabel("Average Salary")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


## 10. Job Role Analysis

In [ ]:
role_summary = df.groupby("Job_Role").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Experience=("Experience_Years", "mean"),
    Avg_Performance=("Performance_Rating", "mean"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).sort_values("Employees", ascending=False)

display(role_summary.round(2))


## 11. Salary & Experience Analysis

In [ ]:
print("Minimum salary:", df["Salary"].min())
print("Median salary:", df["Salary"].median())
print("Average salary:", df["Salary"].mean())
print("Maximum salary:", df["Salary"].max())

plt.figure(figsize=(9,5))
sns.histplot(df["Salary"], bins=20, kde=True)
plt.title("Salary Distribution")
plt.xlabel("Salary")
plt.ylabel("Employees")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(
    data=df,
    x="Experience_Years",
    y="Salary",
    hue="Department",
    alpha=0.7
)
plt.title("Experience vs Salary")
plt.xlabel("Experience (Years)")
plt.ylabel("Salary")
plt.tight_layout()
plt.show()

print("Experience-Salary correlation:",
      round(df["Experience_Years"].corr(df["Salary"]), 3))


## 12. Performance, Satisfaction & Training

In [ ]:
performance_summary = df.groupby("Performance_Rating").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Training_Hours=("Training_Hours", "mean"),
    Avg_Satisfaction=("Job_Satisfaction", "mean"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).reset_index()

display(performance_summary.round(2))


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Performance_Rating", y="Training_Hours")
plt.title("Training Hours by Performance Rating")
plt.xlabel("Performance Rating")
plt.ylabel("Training Hours")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="Job_Satisfaction")
plt.title("Job Satisfaction Distribution")
plt.xlabel("Job Satisfaction")
plt.ylabel("Employees")
plt.tight_layout()
plt.show()


## 13. Overtime & Absenteeism

In [ ]:
overtime_summary = df.groupby("Overtime").agg(
    Employees=("Employee_ID", "count"),
    Avg_Satisfaction=("Job_Satisfaction", "mean"),
    Avg_Work_Life_Balance=("Work_Life_Balance", "mean"),
    Avg_Absenteeism=("Absenteeism_Days", "mean"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
)

display(overtime_summary.round(2))


In [ ]:
absenteeism_dept = df.groupby("Department")["Absenteeism_Days"].mean().sort_values(ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(x=absenteeism_dept.values, y=absenteeism_dept.index)
plt.title("Average Absenteeism by Department")
plt.xlabel("Average Absenteeism Days")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


## 14. Attrition Analysis

In [ ]:
attrition_summary = df.groupby("Attrition").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Experience=("Experience_Years", "mean"),
    Avg_Satisfaction=("Job_Satisfaction", "mean"),
    Avg_Work_Life_Balance=("Work_Life_Balance", "mean"),
    Avg_Absenteeism=("Absenteeism_Days", "mean"),
    Overtime_Rate=("Overtime", lambda x: x.eq("Yes").mean()*100)
)

display(attrition_summary.round(2))


In [ ]:
attrition_department = df.groupby("Department").agg(
    Employees=("Employee_ID", "count"),
    Attrition_Count=("Attrition", lambda x: x.eq("Yes").sum()),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).sort_values("Attrition_Rate", ascending=False)

display(attrition_department.round(2))

plt.figure(figsize=(9,5))
sns.barplot(
    x=attrition_department["Attrition_Rate"],
    y=attrition_department.index
)
plt.title("Attrition Rate by Department")
plt.xlabel("Attrition Rate (%)")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


In [ ]:
# Attrition vs overtime
overtime_attrition = pd.crosstab(
    df["Overtime"],
    df["Attrition"],
    normalize="index"
) * 100

display(overtime_attrition.round(2))

overtime_attrition.plot(kind="bar", figsize=(8,5))
plt.title("Attrition Percentage by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Attrition vs satisfaction
satisfaction_attrition = df.groupby("Job_Satisfaction").agg(
    Employees=("Employee_ID", "count"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
)

display(satisfaction_attrition.round(2))


## 15. Education & Location Analysis

In [ ]:
education_summary = df.groupby("Education").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Experience=("Experience_Years", "mean"),
    Avg_Performance=("Performance_Rating", "mean")
).sort_values("Employees", ascending=False)

display(education_summary.round(2))


In [ ]:
location_summary = df.groupby("Location").agg(
    Employees=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Performance=("Performance_Rating", "mean"),
    Attrition_Rate=("Attrition", lambda x: x.eq("Yes").mean()*100)
).sort_values("Employees", ascending=False)

display(location_summary.round(2))


## 16. Hiring Trend Analysis

In [ ]:
df["Joining_Year"] = df["Joining_Date"].dt.year
df["Joining_Month"] = df["Joining_Date"].dt.month

yearly_hiring = df.groupby("Joining_Year").agg(
    Employees_Joined=("Employee_ID", "count"),
    Avg_Salary=("Salary", "mean"),
    Avg_Experience=("Experience_Years", "mean")
).reset_index()

display(yearly_hiring.round(2))


In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(
    data=yearly_hiring,
    x="Joining_Year",
    y="Employees_Joined",
    marker="o"
)
plt.title("Employee Hiring Trend by Year")
plt.xlabel("Joining Year")
plt.ylabel("Employees Joined")
plt.tight_layout()
plt.show()


In [ ]:
monthly_hiring = (
    df.groupby(df["Joining_Date"].dt.to_period("M"))
      .size()
      .reset_index(name="Employees_Joined")
)

monthly_hiring["Joining_Date"] = monthly_hiring["Joining_Date"].dt.to_timestamp()

plt.figure(figsize=(12,5))
sns.lineplot(
    data=monthly_hiring,
    x="Joining_Date",
    y="Employees_Joined"
)
plt.title("Monthly Hiring Trend")
plt.xlabel("Month")
plt.ylabel("Employees Joined")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 17. Correlation Analysis

In [ ]:
numeric_cols = [
    "Age", "Experience_Years", "Salary",
    "Performance_Rating", "Job_Satisfaction",
    "Work_Life_Balance", "Training_Hours",
    "Absenteeism_Days"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("HR Correlation Matrix")
plt.tight_layout()
plt.show()

display(corr.round(2))


## 18. HR Attention Segmentation

In [ ]:
df["Attention_Group"] = np.select(
    [
        df["Attrition"].eq("Yes"),
        (df["Job_Satisfaction"] <= 2) &
        (df["Work_Life_Balance"] <= 2) &
        df["Overtime"].eq("Yes"),
        (df["Job_Satisfaction"] <= 2) |
        (df["Work_Life_Balance"] <= 2) |
        df["Overtime"].eq("Yes")
    ],
    [
        "Already Left",
        "High Attention",
        "Needs Attention"
    ],
    default="Lower Attention"
)

attention_summary = df["Attention_Group"].value_counts().reset_index()
attention_summary.columns = ["Attention_Group", "Employees"]

display(attention_summary)


In [ ]:
# Employees needing attention
attention_employees = df[
    df["Attention_Group"].isin(["High Attention", "Needs Attention"])
][[
    "Employee_ID", "Department", "Job_Role", "Age", "Salary",
    "Job_Satisfaction", "Work_Life_Balance", "Overtime",
    "Absenteeism_Days", "Attrition", "Attention_Group"
]]

display(attention_employees.head(20))


## 19. Business Insights Checklist

After running the notebook, document findings such as:

1. **Workforce:** Which department has the largest workforce?
2. **Salary:** Which department and job role have the highest average salary?
3. **Attrition:** Which department has the highest attrition rate?
4. **Overtime:** Is attrition higher among overtime employees?
5. **Satisfaction:** Which satisfaction rating is associated with the highest attrition?
6. **Work-life balance:** Which groups have weaker work-life balance?
7. **Performance:** Which departments have the strongest performance?
8. **Training:** How do training hours differ across performance groups?
9. **Absenteeism:** Which departments have the highest absenteeism?
10. **Hiring:** How has employee hiring changed over time?

**Important:** EDA identifies patterns and associations. It does not prove causation.


## 20. Power BI Dashboard Recommendation

### Page 1 — HR Executive Overview
- Total Employees
- Average Salary
- Attrition Rate
- Average Performance
- Average Satisfaction
- Employees by Department
- Employees by Location

### Page 2 — Attrition & Retention
- Attrition by Department
- Attrition vs Overtime
- Attrition vs Job Satisfaction
- Attrition vs Work-Life Balance
- Attrition by Age Group

### Page 3 — Salary & Workforce
- Salary by Department
- Salary by Job Role
- Experience vs Salary
- Salary Distribution
- Education Analysis

### Page 4 — Performance & Well-Being
- Performance by Department
- Training vs Performance
- Satisfaction by Department
- Absenteeism by Department

### Page 5 — Hiring Trends
- Yearly Hiring Trend
- Monthly Hiring Trend
- Average Salary by Joining Year
